# TikTok search-term discovery

This notebook does one thing: it starts with an initial seed term and discovers the search language TikTok associates with it.

```text
seed term
    -> first-order TikTok suggestions
        -> second-order TikTok suggestions
```

The notebook requests TikTok's undocumented web autocomplete endpoint through a visible, persistent Edge session. It records only autocomplete terms and their search lineage. It does not collect video URLs, score trends, download content, or bypass verification challenges.

## Prerequisites

Install the repository dependencies before opening the notebook:

```powershell
pip install -r requirements.txt
```

Microsoft Edge must be installed. The notebook stores its dedicated browser state under `.edge_profile_search_terms/`, which is excluded from Git.

## 1. Configuration

Edit `SEARCH_TERM`. The workflow uses exactly two expansion layers so every second-layer suggestion remains traceable to each first-layer parent that produced it.

In [ ]:
from __future__ import annotations

import json
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display
from playwright.async_api import async_playwright

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "tiktok_video_search.py").exists():
    raise RuntimeError(
        "Launch JupyterLab from the repository root so PROJECT_ROOT contains "
        "tiktok_video_search.py."
    )

OUTPUT_DIR = PROJECT_ROOT / "outputs"
PROFILE_DIR = PROJECT_ROOT / ".edge_profile_search_terms"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_TERM = "creepy tok"
MAX_SUGGESTIONS_PER_QUERY = 12
MAX_FIRST_LAYER_PARENTS = 20
MAX_TOTAL_OCCURRENCES = 200
QUERY_PAUSE_MS = 900
HEADLESS = False

print("Seed term:", SEARCH_TERM)
print("Expansion layers: 2")

In [ ]:
UI_LABELS = {
    "accounts", "discover", "explore", "for you", "hashtags",
    "live", "search", "sounds", "users", "videos", "view more",
}


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def normalize_term(value: str) -> str:
    value = (value or "").replace("#", " ")
    value = re.sub(r"\s+", " ", value).strip(" -\u2013\u2014|:;,.")
    if not 2 <= len(value) <= 80 or "\n" in value:
        return ""
    if value.casefold() in UI_LABELS:
        return ""
    return value


def unique_terms(values: list[str], exclude: set[str] | None = None) -> list[str]:
    excluded = {value.casefold() for value in (exclude or set())}
    seen = set(excluded)
    output = []
    for value in values:
        term = normalize_term(value)
        key = term.casefold()
        if term and key not in seen:
            seen.add(key)
            output.append(term)
    return output


def review_record(
    *,
    term: str,
    order: int,
    first_order_term: str,
    parent: str,
    query_path: list[str],
    source: str,
) -> dict[str, Any]:
    return {
        "term": term,
        "order": order,
        "first_order_term": first_order_term,
        "parent": parent,
        "query_path": " > ".join(query_path),
        "source": source,
        "review_decision": "unreviewed",
        "review_notes": "",
        "collected_at": utc_now_iso(),
    }


def build_discovery_structure(
    seed: str,
    first_layer_items: list[dict[str, str]],
    second_layer_by_parent: dict[str, list[dict[str, str]]],
) -> dict[str, Any]:
    first_records = []
    second_records = []
    branches = []
    edges = []

    for first_item in first_layer_items:
        first_term = normalize_term(first_item.get("term", ""))
        if not first_term:
            continue

        first_record = review_record(
            term=first_term,
            order=1,
            first_order_term=first_term,
            parent=seed,
            query_path=[seed, first_term],
            source=first_item.get("source", "tiktok_autocomplete"),
        )
        first_records.append(first_record)
        edges.append(
            {
                "parent": seed,
                "child": first_term,
                "depth": 1,
                "first_order_term": first_term,
                "query_path": first_record["query_path"],
                "source": first_record["source"],
            }
        )

        branch_children = []
        for second_item in second_layer_by_parent.get(first_term, []):
            second_term = normalize_term(second_item.get("term", ""))
            if not second_term or second_term.casefold() == first_term.casefold():
                continue
            second_record = review_record(
                term=second_term,
                order=2,
                first_order_term=first_term,
                parent=first_term,
                query_path=[seed, first_term, second_term],
                source=second_item.get("source", "tiktok_autocomplete"),
            )
            branch_children.append(second_record)
            second_records.append(second_record)
            edges.append(
                {
                    "parent": first_term,
                    "child": second_term,
                    "depth": 2,
                    "first_order_term": first_term,
                    "query_path": second_record["query_path"],
                    "source": second_record["source"],
                }
            )

        branches.append(
            {
                "first_order_term": first_term,
                "review_decision": "unreviewed",
                "review_notes": "",
                "first_layer": first_record,
                "second_layer": branch_children,
            }
        )

    return {
        "layers": {
            "1": [record["term"] for record in first_records],
            "2": unique_terms([record["term"] for record in second_records]),
        },
        "branches": branches,
        "records": [*first_records, *second_records],
        "edges": edges,
    }


def save_json(path: Path, payload: Any) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    temporary.replace(path)


assert normalize_term("  #BookTok  ") == "BookTok"
assert unique_terms(["BookTok", "booktok", "Book reviews"]) == [
    "BookTok", "Book reviews"
]

duplicate_fixture = build_discovery_structure(
    "seed",
    [
        {"term": "first a", "source": "fixture"},
        {"term": "first b", "source": "fixture"},
    ],
    {
        "first a": [{"term": "shared child", "source": "fixture"}],
        "first b": [{"term": "shared child", "source": "fixture"}],
    },
)
assert duplicate_fixture["branches"][0]["second_layer"][0]["term"] == "shared child"
assert duplicate_fixture["branches"][1]["second_layer"][0]["term"] == "shared child"
assert duplicate_fixture["layers"]["2"] == ["shared child"]
print("Helper and branch-nesting tests passed.")

## 2. Open TikTok in a persistent Edge profile

Sign in manually if required. If TikTok presents a verification challenge, complete it in the visible browser; the notebook does not bypass challenges. If a later cell fails, run the closing cell before restarting the browser cell.

In [ ]:
playwright_instance = None
browser_context = None


async def close_browser_session() -> None:
    global browser_context, playwright_instance
    if browser_context is not None:
        await browser_context.close()
        browser_context = None
    if playwright_instance is not None:
        await playwright_instance.stop()
        playwright_instance = None


playwright_instance = await async_playwright().start()
try:
    browser_context = await playwright_instance.chromium.launch_persistent_context(
        user_data_dir=str(PROFILE_DIR),
        channel="msedge",
        headless=HEADLESS,
        viewport=None,
        locale="en-AU",
        args=["--start-maximized"],
    )
    page = (
        browser_context.pages[0]
        if browser_context.pages
        else await browser_context.new_page()
    )
    await page.goto(
        "https://www.tiktok.com/",
        wait_until="domcontentloaded",
        timeout=90_000,
    )
except Exception:
    await close_browser_session()
    raise

print("Edge is open. Sign in or resolve any visible challenge before continuing.")

## 3. TikTok autocomplete collector

The collector requests the undocumented JSON endpoint used by TikTok's web autocomplete. The visible Edge session supplies normal site context, while the bounded delay and result limits keep the discovery run small. Because this endpoint is undocumented, TikTok can change or remove it without notice.

In [ ]:
TIKTOK_SUGGESTION_ENDPOINT = "https://www.tiktok.com/api/search/general/sug/"


async def get_tiktok_suggestions(query: str) -> list[dict[str, str]]:
    query = normalize_term(query)
    if not query:
        return []

    response = await page.request.get(
        TIKTOK_SUGGESTION_ENDPOINT,
        params={"keyword": query, "aid": "1988"},
        headers={"Referer": "https://www.tiktok.com/"},
        timeout=30_000,
    )
    if not response.ok:
        raise RuntimeError(f"TikTok suggestion request failed: HTTP {response.status}")

    payload = await response.json()
    autocomplete = unique_terms(
        [item.get("content", "") for item in payload.get("sug_list", [])],
        exclude={query},
    )[:MAX_SUGGESTIONS_PER_QUERY]
    await page.wait_for_timeout(QUERY_PAUSE_MS)
    return [
        {"term": term, "source": "tiktok_web_autocomplete"}
        for term in autocomplete
    ]


print("Autocomplete collector ready.")

## 4. Expand the seed term

The first request finds first-order terms. Each retained first-order term then receives one second-order request. A repeated child is queried only once as a unique term but remains recorded beneath every parent that produced it.

In [ ]:
if not SEARCH_TERM or "REPLACE WITH" in SEARCH_TERM.upper():
    raise ValueError("Edit SEARCH_TERM in the configuration cell first.")

seed = normalize_term(SEARCH_TERM)
first_layer_items = (
    await get_tiktok_suggestions(seed)
)[:MAX_FIRST_LAYER_PARENTS]

second_layer_by_parent: dict[str, list[dict[str, str]]] = {}
occurrence_count = len(first_layer_items)

print(f"First layer: {len(first_layer_items)} term(s)")
for parent_index, first_item in enumerate(first_layer_items, start=1):
    parent = first_item["term"]
    remaining = MAX_TOTAL_OCCURRENCES - occurrence_count
    if remaining <= 0:
        break
    print(f"  {parent_index}/{len(first_layer_items)} {parent!r}", end="\r")
    children = (await get_tiktok_suggestions(parent))[:remaining]
    second_layer_by_parent[parent] = children
    occurrence_count += len(children)

discovery = build_discovery_structure(
    seed,
    first_layer_items,
    second_layer_by_parent,
)
layers = discovery["layers"]
branches = discovery["branches"]
records = discovery["records"]
edges = discovery["edges"]

columns = [
    "term", "order", "first_order_term", "parent", "query_path",
    "source", "review_decision", "review_notes", "collected_at",
]
terms_df = pd.DataFrame(records, columns=columns)
terms_for_review_df = terms_df.sort_values(
    ["first_order_term", "order", "term"],
    key=lambda column: column.astype(str).str.casefold(),
).reset_index(drop=True)

if terms_for_review_df.empty:
    terms_for_review_df.insert(0, "display_term", pd.Series(dtype="object"))
    terms_for_review_df.insert(0, "layer", pd.Series(dtype="object"))
else:
    terms_for_review_df.insert(
        0,
        "display_term",
        terms_for_review_df.apply(
            lambda row: row["term"]
            if int(row["order"]) == 1
            else f"    -> {row['term']}",
            axis=1,
        ),
    )
    terms_for_review_df.insert(
        0,
        "layer",
        terms_for_review_df["order"].map(
            {1: "FIRST_LAYER", 2: "SECOND_LAYER"}
        ),
    )

print(f"\nFirst-layer unique terms: {len(layers['1'])}")
print(f"Second-layer unique terms: {len(layers['2'])}")
print(f"Recorded parent-child occurrences: {len(records)}")

In [ ]:
branch_summary_df = pd.DataFrame(
    [
        {
            "first_order_term": branch["first_order_term"],
            "second_layer_terms": len(branch["second_layer"]),
            "total_branch_terms": 1 + len(branch["second_layer"]),
        }
        for branch in branches
    ]
)
display(branch_summary_df)
display(terms_for_review_df.head(15))

## 5. Export search terms

The CSV is the human review surface. It retains repeated child occurrences under their respective parents. The JSON preserves the same complete branch nesting for the handoff notebook.

In [ ]:
latest_csv_path = OUTPUT_DIR / "latest_search_terms.csv"
latest_json_path = OUTPUT_DIR / "latest_search_terms.json"

terms_for_review_df.to_csv(latest_csv_path, index=False, encoding="utf-8-sig")
payload = {
    "schema_version": "2.0",
    "file_purpose": (
        "Two-level TikTok web-autocomplete expansion for an initial seed term."
    ),
    "search_term": seed,
    "expansion_layers": 2,
    "collected_at": utc_now_iso(),
    "layers": layers,
    "branches": branches,
    "records": records,
    "edges": edges,
}
save_json(latest_json_path, payload)

print(f"Exported review CSV: {latest_csv_path}")
print(f"Exported nested JSON: {latest_json_path}")
print(f"First layer: {len(layers['1'])} unique terms")
print(f"Second layer: {len(layers['2'])} unique terms")

## 6. Close Edge

Run this after exporting, or after an error in a later cell. The dedicated profile remains available for later runs.

In [ ]:
await close_browser_session()
print("Browser closed.")

## Interpretation

First-order terms are the closest TikTok-native expansions of the seed. Second-order terms expose modifiers and adjacent language while retaining the first-order branch that generated them.

`layers` contains globally unique terms for compact summaries. `branches`, `records`, and the CSV preserve every parent-child occurrence, including a second-layer term that appears beneath multiple first-layer parents.

Use `review_decision` for human triage: `analyse_videos` for terms worth passing to video discovery, `exclude` for clear semantic drift, and `uncertain` when a small manual TikTok check is needed. Record the reason in `review_notes`. The notebook deliberately does not infer these decisions from wording alone.